# 📓 Semana 13 · Dia 5 — Monitoramento contínuo de RAG: qualidade, custo e segurança

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Monitor de RAG com 4 frentes |

---


## 📖 Teoria — As 4 frentes de monitoramento

| Frente | Métricas | Ferramenta |
|---|---|---|
| **Qualidade** | faithfulness, relevance, recall | mlflow.evaluate agendado |
| **Operacional** | latência, erros, taxa de sucesso | endpoint logs |
| **Custo** | tokens, $ por pergunta, cache hit | gateway/billing |
| **Segurança** | PII na resposta, prompts maliciosos | guardrails/logs |


### 💻 Na prática — Qualidade contínua

Monte o notebook de avaliação agendada.


In [ ]:
# Avaliação agendada (job diário)
import mlflow
golden = spark.table("workspace.prata.golden_set").toPandas()
respostas = []
for q in golden["pergunta"]:
    respostas.append(rag.invoke({"input": q})["answer"])
golden["response"] = respostas
with mlflow.start_run(run_name="avaliacao_diaria"):
    mlflow.evaluate(
        data=golden[["pergunta", "response"]],
        targets=golden["resposta_esperada"],
        model_type="databricks-agent",
        extra_metrics=[mlflow.metrics.genai.faithfulness(),
                       mlflow.metrics.genai.answer_relevance()])
print("Avaliação diária registrada (agende como job).")

### 💻 Na prática — Custo por pergunta

Estime o custo por chamada (tokens de entrada/saída).


In [ ]:
# Estimativa de custo por pergunta
def estimar_custo(tokens_in, tokens_out, preco_in=3e-6, preco_out=12e-6):
    return round(tokens_in * preco_in + tokens_out * preco_out, 4)
print("Custo estimado por pergunta (ex.): $",
      estimar_custo(1200, 150))
print("Meta: custo < $0.01/pergunta com cache + modelo barato.")

### 💻 Na prática — Segurança

Verifique PII nas respostas e registre prompts sensíveis.


In [ ]:
# Checagem simples de PII na resposta
import re
def detecta_pii(texto):
    email = re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", texto)
    cpf = re.findall(r"\d{3}\.\d{3}\.\d{3}-\d{2}", texto)
    return {"email": email, "cpf": cpf}
print(detecta_pii("Contato: ana@empresa.com · CPF 123.456.789-00"))
print("Se o RAG vazar PII, aplique masking/guardrails (Semana 15).")

> 🎯 **Dica de prova**: GenAI Assoc (Evaluation & Monitoring ~20%): monitorar 4 frentes é o padrão. Pergunta: 'quais métricas monitorar num RAG?' → qualidade, custo, latência, segurança.


## 🎯 Exercícios de fixação

**1.** Monte um painel com as 4 frentes (queries SQL).

**2.** O que um aumento de custo por pergunta indica?

**3.** Como detectar vazamento de PII automaticamente?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Painel

4 queries: métricas do mlflow (qualidade), latência do endpoint (operacional), tokens/$ (custo), logs com regex PII (segurança).

**2.** Custo subiu

Contexto maior (chunks), modelo caro, cache miss — revise chunking/roteamento.

**3.** PII

Regex + modelo classificador na resposta; alerta e masking automático (Semana 15).



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*